In [1]:

import ee
import geemap

# 1. Autenticación e Inicialización (Solo necesitas autenticar la primera vez)
try:
    ee.Initialize()
except Exception as e:
    ee.Authenticate()
    ee.Initialize(project = "ee-my-usenvdi" )


# 2. Definir el Área de Interés (AOI) - Ejemplo: Una región en el sur de Europa
aoi = ee.Geometry.Point([-3.703790, 40.416775]).buffer(100000) # 100km alrededor de Madrid

# 3. Cargar el dataset ERA5-Land (Temperatura diaria agregada)
# Seleccionamos la temperatura máxima a 2 metros de altura
dataset = ee.ImageCollection("ECMWF/ERA5_LAND/DAILY_AGGR") \
            .filterBounds(aoi) \
            .select('temperature_2m')

# 4. Definir el periodo de la ola de calor (Ejemplo: Julio de 2023)
fecha_inicio = '2023-07-01'
fecha_fin = '2023-07-31'
dataset_evento = dataset.filterDate(fecha_inicio, fecha_fin)

# 5. Función para convertir Kelvin a Celsius
def kelvin_a_celsius(image):
    # ERA5 viene en Kelvin. Restamos 273.15 para pasar a Celsius.
    celsius = image.subtract(273.15).rename('Temp_Celsius')
    return celsius.copyProperties(image, ['system:time_start'])

# Aplicar la conversión a toda la colección
temp_celsius = dataset_evento.map(kelvin_a_celsius)

# 6. Calcular la temperatura máxima absoluta de ese mes para cada píxel
temp_max_evento = temp_celsius.max().clip(aoi)

# 7. Visualización con Geemap
Map = geemap.Map()
Map.centerObject(aoi, 7)

# Parámetros de visualización (de 25°C a 45°C)
vis_params = {
    'min': 25,
    'max': 45,
    'palette': ['blue', 'cyan', 'green', 'yellow', 'orange', 'red', 'darkred']
}

Map.addLayer(temp_max_evento, vis_params, 'Temperatura Máxima (Julio 2023)')
Map.addLayerControl()

# Mostrar el mapa (si estás en un entorno Notebook)
Map


Map(center=[40.416877572114664, -3.70378626390922], controls=(WidgetControl(options=['position', 'transparent_…